pip install pandas

pip install numpy

pip install scikit-learn

pip install spacy

python -m pip install -U pip
python -m pip install -U spacy
python -m spacy download en_core_web_sm



In [61]:
#import pandas
import pandas as pd
import numpy as np
import spacy
import re
import random
from sklearn.model_selection import train_test_split
from collections import Counter

In [11]:
# Load SpaCy for text processing
nlp = spacy.load('en_core_web_sm')

In [14]:
#read the dataset
data = pd.read_csv(r'DDI_data.csv')

In [15]:
# Using a method to control the rows and columns truncation
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 150)

In [16]:
#checking the dataset
data.head(10)

,drug1_id,drug2_id,drug1_name,drug2_name,interaction_type
0,DB00006,DB00346,Bivalirudin,Alfuzosin,serum concentration
1,DB00006,DB13783,Bivalirudin,Acemetacin,risk or severity of bleeding
2,DB00006,DB06605,Bivalirudin,Apixaban,anticoagulant activities
3,DB00006,DB06695,Bivalirudin,Dabigatran etexilate,anticoagulant activities
4,DB00006,DB09075,Bivalirudin,Edoxaban,anticoagulant activities
5,DB00006,DB06228,Bivalirudin,Rivaroxaban,anticoagulant activities
6,DB00006,DB00227,Bivalirudin,Lovastatin,serum concentration
7,DB00006,DB09030,Bivalirudin,Vorapaxar,risk or severity of adverse effects
8,DB00006,DB00683,Bivalirudin,Midazolam,serum concentration
9,DB00006,DB00641,Bivalirudin,Simvastatin,serum concentration


In [17]:
#checking the columns of the dataset
list(data.columns)

['drug1_id', 'drug2_id', 'drug1_name', 'drug2_name', 'interaction_type']

In [18]:
# Display the number of instances and features
data.shape

(222696, 5)

In [19]:
#checking the datatypes
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 222696 entries, 0 to 222695
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   drug1_id          222696 non-null  object
 1   drug2_id          222696 non-null  object
 2   drug1_name        222696 non-null  object
 3   drug2_name        222696 non-null  object
 4   interaction_type  222696 non-null  object
dtypes: object(5)
memory usage: 8.5+ MB


In [20]:
#describing data
data.describe()

,drug1_id,drug2_id,drug1_name,drug2_name,interaction_type
count,222696,222696,222696,222696,222696
unique,1751,1789,1751,1789,114
top,DB00176,DB11672,Fluvoxamine,Curcumin,risk or severity of adverse effects
freq,925,876,925,876,62767


In [21]:
#checking null values
data.isnull().sum()

drug1_id            0
drug2_id            0
drug1_name          0
drug2_name          0
interaction_type    0
dtype: int64

In [22]:
null_columns = data.columns[data.isnull().any()]
print("Columns with null values:")
print(null_columns)

Columns with null values:
Index([], dtype='object')


In [23]:
data['drug1_id'].unique()

array(['DB00006', 'DB00014', 'DB00027', ..., 'DB14011', 'DB14019',
       'DB14033'], shape=(1751,), dtype=object)

In [ ]:
#checking unique values
data['interaction_type'].unique()

In [25]:
def map_interaction_severity(interaction_type):
    """
    Map interaction descriptions to severity levels
    Severity: 0=None, 1=Mild, 2=Moderate, 3=Severe
    """
    interaction_lower = interaction_type.lower()
    
    # Severe interactions (life-threatening)
    severe_keywords = [
         "risk or severity",
    "severe",
    "fatal",
    "death",
    "bleeding",
    "hemorrhage",
    "cardiac arrest",
    "torsade",
    "qt prolongation",
    "ventricular arrhythmias",
    "respiratory depressant",
    "renal failure",
    "rhabdomyolysis",
    "myoglobinuria",
    "seizure",
    "convulsion",
    "neurotoxic",
    "hepatotoxic",
    "pulmonary toxicity",
    "myelosuppression",
    "leukopenia",
    "angioedema",
    "teratogenic"
    ]
    
    # Moderate interactions (require monitoring)
    moderate_keywords = [
        "increased",
    "decreased",
    "serum concentration",
    "active metabolites",
    "metabolism",
    "excretion",
    "clearance",
    "bioavailability",
    "protein binding",
    "hypotension",
    "hypertension",
    "bradycardia",
    "tachycardic",
    "cns depressant",
    "sedative",
    "nephrotoxic",
    "hepatotoxic activities",
    "hypoglycemic",
    "hyperglycemic",
    "hyperkalemia",
    "hypokalemia",
    "hyponatremia",
    "fluid retention",
    "edema"
    ]
    
    # Mild interactions (minimal clinical significance)
    mild_keywords = [
        "absorption",
    "therapeutic efficacy",
    "pharmacokinetic",
    "diagnostic agent",
    "antiplatelet activities",
    "vasodilatory activities",
    "bronchodilatory activities",
    "analgesic activities",
    "minor",
    "slight",
    "may increase",
    "may decrease"
    ]
    
    # Check for severe
    if any(keyword in interaction_lower for keyword in severe_keywords):
        return 3
    
    # Check for moderate
    if any(keyword in interaction_lower for keyword in moderate_keywords):
        return 2
    
    # Check for mild
    if any(keyword in interaction_lower for keyword in mild_keywords):
        return 1
    
    # Default to moderate if unclear
    return 2

In [28]:
def create_severity_labels(data):
    """Add severity column to dataframe"""
    data['severity'] = data['interaction_type'].apply(map_interaction_severity)
    
    print("\nSeverity Distribution:")
    print(data['severity'].value_counts().sort_index())
    print("\nSeverity Mapping:")
    print("0: None, 1: Mild, 2: Moderate, 3: Severe")

    return data

In [30]:
# ===================================================================
# 3. GENERATE SYNTHETIC PRESCRIPTION TEXTS
# ===================================================================

# Sample data for synthetic generation
DEMOGRAPHICS = [
    "25F", "30M", "45F", "52M", "60F", "68M", "35F", "40M",
    "55F", "62M", "28F", "33M", "48F", "58M", "70F", "75M"
]

DOSAGES = {
    'tablet': ['25mg', '50mg', '75mg', '100mg', '200mg', '500mg'],
    'injection': ['5mg', '10mg', '20mg', '50mg'],
    'liquid': ['5ml', '10ml', '15ml', '20ml']
}

FREQUENCIES = [
    'once daily', 'twice daily', 'three times daily',
    'every 6 hours', 'every 8 hours', 'every 12 hours',
    'as needed', 'before meals', 'after meals'
]

In [32]:
def generate_prescription_text(drug_list, demographics=None, include_dosage=True):
    """Generate realistic prescription text"""
    if demographics is None:
        demographics = random.choice(DEMOGRAPHICS)
    
    prescription_parts = [f"Patient: {demographics}"]
    
    drug_entries = []
    for drug in drug_list:
        if include_dosage and random.random() > 0.3:
            dosage = random.choice(DOSAGES['tablet'])
            frequency = random.choice(FREQUENCIES)
            drug_entries.append(f"{drug} {dosage} {frequency}")
        else:
            drug_entries.append(drug)
    
    prescription_parts.append(f"Drugs: {', '.join(drug_entries)}")
    
    return '. '.join(prescription_parts) + '.'

In [ ]:
def create_synthetic_dataset(data, n_samples=2000):
    """
    Generate synthetic prescription texts with labels
    Creates both interacting and non-interacting combinations
    """
    synthetic_data = []
    
    # Get unique drugs
    all_drugs = list(set(data['drug1_name'].unique().tolist() + 
                         data['drug2_name'].unique().tolist()))
    
    # Create interaction lookup
    interaction_dict = {}
    for _, row in data.iterrows():
        key = tuple(sorted([row['drug1_name'], row['drug2_name']]))
        interaction_dict[key] = {
            'type': row['interaction_type'],
            'severity': row['severity']
        }
    
    # Generate samples with interactions (50%)
    n_with_interaction = n_samples // 2
    print(f"\nGenerating {n_with_interaction} samples WITH interactions...")
    
    sampled_interactions = data.sample(n=min(n_with_interaction, len(df)))
    
    for _, row in sampled_interactions.iterrows():
        drugs = [row['drug1_name'], row['drug2_name']]
        
        # Sometimes add a third drug
        if random.random() > 0.5:
            drugs.append(random.choice(all_drugs))
        
        text = generate_prescription_text(drugs)
        
        synthetic_data.append({
            'prescription_text': text,
            'drugs': drugs,
            'has_interaction': 1,
            'severity': row['severity'],
            'interaction_type': row['interaction_type']
        })
    
    # Generate samples without known interactions (50%)
    n_without_interaction = n_samples - n_with_interaction
    print(f"Generating {n_without_interaction} samples WITHOUT interactions...")
    
    for _ in range(n_without_interaction):
        # Select 2-4 random drugs
        n_drugs = random.randint(2, 4)
        drugs = random.sample(all_drugs, n_drugs)
        
        # Check if any pair has interaction
        has_interaction = False
        max_severity = 0
        interaction_type = 'none'
        
        for i in range(len(drugs)):
            for j in range(i+1, len(drugs)):
                key = tuple(sorted([drugs[i], drugs[j]]))
                if key in interaction_dict:
                    has_interaction = True
                    if interaction_dict[key]['severity'] > max_severity:
                        max_severity = interaction_dict[key]['severity']
                        interaction_type = interaction_dict[key]['type']
        
        text = generate_prescription_text(drugs)
        
        synthetic_data.append({
            'prescription_text': text,
            'drugs': drugs,
            'has_interaction': 1 if has_interaction else 0,
            'severity': max_severity if has_interaction else 0,
            'interaction_type': interaction_type if has_interaction else 'none'
        })
    
    synthetic_df = pd.DataFrame(synthetic_data)
    
    print("\nSynthetic Dataset Created!")
    print(f"Total samples: {len(synthetic_df)}")
    print(f"\nInteraction Distribution:")
    print(synthetic_df['has_interaction'].value_counts())
    print(f"\nSeverity Distribution:")
    print(synthetic_df['severity'].value_counts().sort_index())
    
    return synthetic_df

In [33]:
# ===================================================================
# 4. TEXT PREPROCESSING PIPELINE
# ===================================================================

def preprocess_text(text):
    """
    Comprehensive text preprocessing pipeline
    - Lowercase normalization
    - Remove special characters
    - Clean extra whitespace
    """
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # Remove special characters but keep basic punctuation for sentence structure
    text = re.sub(r'[^a-zA-Z0-9\s\.,:]', '', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

In [34]:
def tokenize_with_spacy(text):
    """Tokenize text using SpaCy"""
    doc = nlp(text)
    tokens = [token.text for token in doc if not token.is_stop and not token.is_punct]
    return tokens

def preprocess_dataset(df):
    """Apply preprocessing to entire dataset"""
    print("\nApplying text preprocessing...")
    
    df['preprocessed_text'] = df['prescription_text'].apply(preprocess_text)
    df['tokens'] = df['preprocessed_text'].apply(tokenize_with_spacy)
    df['token_count'] = df['tokens'].apply(len)
    
    print("Preprocessing complete!")
    print(f"\nAverage token count: {df['token_count'].mean():.2f}")
    print(f"Max token count: {df['token_count'].max()}")
    print(f"Min token count: {df['token_count'].min()}")
    
    return df

In [ ]:
# ===================================================================
# 5. CREATE BALANCED DATASET
# ===================================================================

def balance_dataset(data, target_column='severity'):
    """Create balanced dataset by severity level"""
    print("\nBalancing dataset by severity level...")
    
    # Count samples per class
    class_counts = data[target_column].value_counts()
    print("Original distribution:")
    print(class_counts)
    
    # Find minimum class count
    min_count = class_counts.min()
    
    # Sample equally from each class
    balanced_columns = []
    for severity in data[target_column].unique():
        class_df = data[data[target_column] == severity]
        sampled = class_df.sample(n=min(len(class_df), min_count), random_state=42)
        balanced_columns.append(sampled)
    
    balanced_column = pd.concat(balanced_columns, ignore_index=True)
    balanced_column = balanced_column.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print("\nBalanced distribution:")
    print(balanced_column[target_column].value_counts().sort_index())
    
    return balanced_column

In [43]:
# ===================================================================
# 6. TRAIN-VAL-TEST SPLIT
# ===================================================================

def split_dataset(data, train_size=0.7, val_size=0.15, test_size=0.15):
    """Split dataset into train, validation, and test sets"""
    assert abs(train_size + val_size + test_size - 1.0) < 0.01
    
    # First split: train and temp (val + test)
    train_data, temp_data = train_test_split(
        data, 
        test_size=(1 - train_size),
        random_state=42,
        stratify=data['severity']
    )
    
    # Second split: val and test
    val_ratio = val_size / (val_size + test_size)
    val_df, test_df = train_test_split(
        temp_data,
        test_size=(1 - val_ratio),
        random_state=42,
        stratify=temp_data['severity']
    )
    
    print("\nDataset Split:")
    print(f"Train: {len(train_data)} samples ({len(train_data)/len(data)*100:.1f}%)")
    print(f"Val: {len(val_df)} samples ({len(val_df)/len(data)*100:.1f}%)")
    print(f"Test: {len(test_df)} samples ({len(test_df)/len(data)*100:.1f}%)")

    return train_data, val_df, test_df

In [44]:
# ===================================================================
# 7. MAIN EXECUTION PIPELINE
# ===================================================================

def main(data, output_dir='./data'):
    """
    Main preprocessing pipeline
    
    Args:
        ddi_filepath: Path to DDI CSV file
        output_dir: Directory to save processed data
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    print("="*70)
    print("DRUG INTERACTION DATA PREPROCESSING PIPELINE")
    print("="*70)
    
    # Step 2: Create severity labels
    print("\n[2/7] Mapping interaction types to severity levels...")
    data = create_severity_labels(data)
    
    # Step 3: Generate synthetic prescription texts
    print("\n[3/7] Generating synthetic prescription texts...")
    synthetic_data = create_synthetic_dataset(data, n_samples=2000)
    
    # Step 4: Preprocess text
    print("\n[4/7] Preprocessing text...")
    synthetic_data = preprocess_dataset(synthetic_data)
    
    # Step 5: Balance dataset
    print("\n[5/7] Balancing dataset...")
    balanced_data = balance_dataset(synthetic_data)
    
    # Step 6: Split dataset
    print("\n[6/7] Splitting dataset...")
    train_data, val_data, test_data = split_dataset(balanced_data)
    
    # Step 7: Save processed data
    print("\n[7/7] Saving processed datasets...")
    train_data.to_csv(f'{output_dir}/train_data.csv', index=False)
    val_data.to_csv(f'{output_dir}/val_data.csv', index=False)
    test_data.to_csv(f'{output_dir}/test_data.csv', index=False)
    data.to_csv(f'{output_dir}/data_processed.csv', index=False)
    
    print(f"\n✓ All files saved to {output_dir}/")
    print("\nFiles created:")
    print(f"  - train_data.csv ({len(train_data)} samples)")
    print(f"  - val_data.csv ({len(val_data)} samples)")
    print(f"  - test_data.csv ({len(test_data)} samples)")
    print(f"  - data_processed.csv ({len(data)} samples)")
    
    # Display sample
    print("\n" + "="*70)
    print("SAMPLE PREPROCESSED DATA:")
    print("="*70)
    print("\nExample from training set:")
    sample = train_data.sample(1).iloc[0]
    print(f"\nOriginal Text: {sample['prescription_text']}")
    print(f"Preprocessed: {sample['preprocessed_text']}")
    print(f"Drugs: {sample['drugs']}")
    print(f"Severity: {sample['severity']}")
    print(f"Has Interaction: {sample['has_interaction']}")
    
    return train_data, val_data, test_data, data

In [45]:
# ===================================================================
# USAGE EXAMPLE
# ===================================================================

if __name__ == "__main__":    
    # Run the complete pipeline
    train_df, val_df, test_df, ddi_df = main(data)
    
    print("\n" + "="*70)
    print("PREPROCESSING COMPLETE!")
    print("="*70)
    print("\nYou can now proceed to Step 2: Drug Entity Recognition (NER)")

DRUG INTERACTION DATA PREPROCESSING PIPELINE

[2/7] Mapping interaction types to severity levels...

Severity Distribution:
severity
1     16396
2    139369
3     66931
Name: count, dtype: int64

Severity Mapping:
0: None, 1: Mild, 2: Moderate, 3: Severe

[3/7] Generating synthetic prescription texts...

Generating 1000 samples WITH interactions...
Generating 1000 samples WITHOUT interactions...

Synthetic Dataset Created!
Total samples: 2000

Interaction Distribution:
has_interaction
1    1310
0     690
Name: count, dtype: int64

Severity Distribution:
severity
0    690
1     92
2    788
3    430
Name: count, dtype: int64

[4/7] Preprocessing text...

Applying text preprocessing...
Preprocessing complete!

Average token count: 13.31
Max token count: 25
Min token count: 5

[5/7] Balancing dataset...

Balancing dataset by severity level...
Original distribution:
severity
2    788
0    690
3    430
1     92
Name: count, dtype: int64

Balanced distribution:
severity
0    92
1    92
2    9

In [46]:
data.head()

,drug1_id,drug2_id,drug1_name,drug2_name,interaction_type,severity
0,DB00006,DB00346,Bivalirudin,Alfuzosin,serum concentration,2
1,DB00006,DB13783,Bivalirudin,Acemetacin,risk or severity of bleeding,3
2,DB00006,DB06605,Bivalirudin,Apixaban,anticoagulant activities,2
3,DB00006,DB06695,Bivalirudin,Dabigatran etexilate,anticoagulant activities,2
4,DB00006,DB09075,Bivalirudin,Edoxaban,anticoagulant activities,2


In [ ]:
# importing a Python package to visualise variables.
import plotly.express as px

In [74]:
#checking anythings histogram
Grade_fig = px.histogram(data, x='severity')
Grade_fig.show(renderer="browser") 

In [50]:
# function to find outliers
def find_outliers_IQR(data):

  q1=data.quantile(0.25)

  q3=data.quantile(0.75)

  IQR=q3-q1

  outliers = data[((data < (q1 - 1.5 * IQR)) | (data > (q3 + 1.5 * IQR)))]

  return outliers

In [51]:
# Finding outliers for anything
outliers = find_outliers_IQR(data['severity'])

print("number of outliers: "+ str(len(outliers)))

outliers

number of outliers: 0


Series([], Name: severity, dtype: int64)

In [57]:
data.describe(include='object')

,drug1_id,drug2_id,drug1_name,drug2_name,interaction_type
count,222696,222696,222696,222696,222696
unique,1751,1789,1751,1789,114
top,DB00176,DB11672,Fluvoxamine,Curcumin,risk or severity of adverse effects
freq,925,876,925,876,62767


In [56]:
data.describe(include='int')

,severity
count,222696.000000
mean,2.226924
std,0.568050
min,1.000000
25%,2.000000
50%,2.000000
75%,3.000000
max,3.000000
